# Data Pull

Pulls all court cases mentioning retired athletes 退役运动员 from China Judgements Online (https://wenshu.court.gov.cn) to a DataFrame. Then exports DataFrame as CSV file "00_raw_cases.csv" to user's working directory (file already available under the "date" directory in repo).

## Imports

In [1]:
# for scrape via CSS attributes
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "selenium", "pandas", "webdriver-manager", "ipywidgets"])
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

import pandas as pd
import time, re
import matplotlib.pyplot as plt
from urllib.parse import urljoin
from IPython.display import display
import os # for exporting to csv


## print mult things
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## Config Scraper Params
for keywords, query url (COMBINED_QUERY) etc.

In [3]:
KEYWORDS     = ["退役运动员"]  # all ANDed in one search
MAX_PAGES    = None        # set to None to scrape all pages. modify for testing.
LOGIN_WAIT   = 40       # seconds to wait for manual login

BASE_URL      = "https://wenshu.court.gov.cn"
SEARCH_URL    = f"{BASE_URL}/website/wenshu/181029CR4M5A62CH/index.html"
COMBINED_QUERY = " ".join(KEYWORDS)

print(f"Search query : '{COMBINED_QUERY}'")
print(f"Max pages    : {MAX_PAGES}")

Search query : '退役运动员'
Max pages    : None


In [4]:
# scraper setup

def get_driver(headless=False):
    options = webdriver.ChromeOptions()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    driver.execute_script(
        "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    )
    return driver


def wait_for(driver, selector, timeout=15):
    return WebDriverWait(driver, timeout).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, selector))
    )

## Manual Web Log In

Username: 13671006853
Password: Wenshu.Court.Gov0!

In [5]:
# open external browser page
driver = get_driver(headless=False)
driver.get(SEARCH_URL)

## Scrape All Pages & Save to df

In [6]:
# enter keywords on page
def search_keywords():
    search_box = driver.find_element(By.CSS_SELECTOR, "input.searchKey")
    driver.execute_script("arguments[0].scrollIntoView(true);", search_box)
    
    time.sleep(0.5)
    search_box.click()
    search_box.clear()
    search_box.send_keys(COMBINED_QUERY)
    print(f"Typed: '{COMBINED_QUERY}'")
    
    search_btn = driver.find_element(By.CSS_SELECTOR, "div.search-rightBtn")
    search_btn.click()
    time.sleep(3)
    first_query_page = driver.current_url
    
    print(f"Current Query Page Timestamp: {first_query_page}")


search_keywords()

Typed: '退役运动员'
Current Query Page Timestamp: https://wenshu.court.gov.cn/website/wenshu/181217BMTKHNT2W0/index.html?pageId=ad46243e31bd1bdabf49df05aa171d03&s21=%E9%80%80%E5%BD%B9%E8%BF%90%E5%8A%A8%E5%91%98


In [7]:

# Paginate through all results, fetch full text per case via new tab, save to df

def fetch_full_text(href):
    """Open href in a new tab, scrape div.PDF_pox, then close the tab."""
    search_tab = driver.current_window_handle
    try:
        driver.execute_script("window.open(arguments[0]);", href)
        driver.switch_to.window(driver.window_handles[-1])
        pdf_div = WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.PDF_pox"))
        )
        text = pdf_div.text.strip()
    except TimeoutException:
        print(f"    Timeout fetching full text: {href}")
        text = None
    except Exception as e:
        print(f"    Error fetching full text: {e}")
        text = None
    finally:
        driver.close()
        driver.switch_to.window(search_tab)  # always return to search tab
    return text


all_info = []
page = 1

while True:
    if MAX_PAGES and page > MAX_PAGES:
        print(f"Reached page limit ({MAX_PAGES}).")
        break

    print(f"\n <<<<<<<<<< Looking at Page {page} >>>>>>>>>>>>>> ", end=" ")
    
    try:
        wait_for(driver, "div.LM_list")
    except TimeoutException:
        print("No results or timeout.")
        break
    
    cases = driver.find_elements(By.CSS_SELECTOR, "div.LM_list")
    print(f"found {len(cases)} cases")
    
    for idx, case in enumerate(cases):
        try:
            a = case.find_element(By.CSS_SELECTOR, "a.caseName")
            href = urljoin(driver.current_url, a.get_attribute("href"))
            print(f"  [{idx+1}/{len(cases)}] {a.text.strip()[:40]}...")
            all_info.append({
                "case_name"  : a.text.strip(),
                "case_type"  : case.find_element(By.CSS_SELECTOR, "div.labelTwo").text.strip(),
                "court"      : case.find_element(By.CSS_SELECTOR, "span.slfyName").text.strip(),
                "date"       : case.find_element(By.CSS_SELECTOR, "span.cprq").text.strip(),
                "case_href"  : href,
                "full_text"  : fetch_full_text(href)
            })
            time.sleep(1.5)  # polite delay after each case
        except NoSuchElementException:
            continue 

    # advance page
    try:
        if driver.find_elements(By.CSS_SELECTOR, "a.disabled pageButton"):
            print("Last page reached.")
            break
            
        next_btn = next(
            el for el in driver.find_elements(By.CSS_SELECTOR, "a.pageButton") if el.text.strip() == "下一页"
        )
        next_btn.click()
        time.sleep(2)
        page += 1
    except:
        print("reached last page.")
        break

df = pd.DataFrame(all_info)



 <<<<<<<<<< Looking at Page 1 >>>>>>>>>>>>>>  

<selenium.webdriver.remote.webelement.WebElement (session="a2c9aa7fc54e60a659f3fba43e28c409", element="f.5317EA51AE2ADCF357AC8A37E323B002.d.DD5E6A3656AB4B8DDDEBFCDA189525DA.e.7509")>

found 5 cases
  [1/5] 北京丰冠体育有限公司等与中国滑雪协会等合同纠纷二审民事判决书...
  [2/5] 刘慷、吉林省射击射箭运动管理中心人事争议民事再审民事裁定书...
  [3/5] 刘小宝、赵效男等买卖合同纠纷民事申请再审审查民事裁定书...
  [4/5] 马晓三、云南省人民检察院劳动争议再审民事判决书...
  [5/5] 李素萍与北京市木樨园体育运动技术学校劳动争议再审审查与审判监督民事裁定书...

 <<<<<<<<<< Looking at Page 2 >>>>>>>>>>>>>>  

<selenium.webdriver.remote.webelement.WebElement (session="a2c9aa7fc54e60a659f3fba43e28c409", element="f.5317EA51AE2ADCF357AC8A37E323B002.d.DD5E6A3656AB4B8DDDEBFCDA189525DA.e.43951")>

found 5 cases
  [1/5] 马俊岭等十三人故意伤害二审刑事附带民事判决书...
  [2/5] 再审申请人张明因与被申请人甘肃省自行车训练管理中心人事纠纷申请再审一案民事裁定书...
  [3/5] 李某猛、某商业保理有限公司）、原审第三人某二有限公司教育培训合同纠纷二审民事判决...
  [4/5] 张某;某学校人事争议二审民事裁定书...
  [5/5] 陈某翔与黄某铭民间借贷纠纷二审民事判决书...

 <<<<<<<<<< Looking at Page 3 >>>>>>>>>>>>>>  

...

found 5 cases
  [1/5] 孙飞燕与北京市芦城体育运动技术学校劳动争议一审民事判决书...
  [2/5] 周雪与广东省足球运动中心劳动争议一审民事判决书...
  [3/5] 冯甲与上海星扬体育文化发展有限公司、上海市浦东新区上南二村小学生命权、健康权、身...
  [4/5] 张明与甘肃省自行车训练管理中心劳动争议一审民事判决书...
  [5/5] 原告付某与被告佳木斯市某局劳动争议纠纷一案的判决书...

 <<<<<<<<<< Looking at Page 16 >>>>>>>>>>>>>>  

<selenium.webdriver.remote.webelement.WebElement (session="a2c9aa7fc54e60a659f3fba43e28c409", element="f.5317EA51AE2ADCF357AC8A37E323B002.d.DD5E6A3656AB4B8DDDEBFCDA189525DA.e.560554")>

found 5 cases
  [1/5] 闫壮与延边体育运动管理中心、延边朝鲜族自治州体育局劳动争议纠纷一案民事裁定书...
  [2/5] 詹永安与北京市体育局其他一审行政裁定书...
  [3/5] 原告纪程诉通化体育运动学校劳动争议一案一审民事判决书...
  [4/5] 王钰与被告四川省运动技术学院劳动争议纠纷一案一审民事判决书...
  [5/5] 胡容儿与张建雄、吴佳丽等道路交通事故人身损害赔偿纠纷一审民事判决书...

 <<<<<<<<<< Looking at Page 17 >>>>>>>>>>>>>>  

<selenium.webdriver.remote.webelement.WebElement (session="a2c9aa7fc54e60a659f3fba43e28c409", element="f.5317EA51AE2ADCF357AC8A37E323B002.d.DD5E6A3656AB4B8DDDEBFCDA189525DA.e.597389")>

found 1 cases
  [1/1] 李雁与陕西省体育训练中心人事争议一审民事判决书...
reached last page.


## Export to CSV

In [1]:
df.to_csv("../data/00_raw_cases.csv", index = False)
print(f"output CSV file saved to {os.getcwd()}")

NameError: name 'df' is not defined